In [4]:
!pip3 install -q --upgrade pip
!pip3 install -q pandas matplotlib seaborn openpyxl requests netCDF4 pandas

In [5]:
import requests
from datetime import datetime

In [2]:
def download_daymet_data(region: str, variable: str, year: int, boundaries: dict, time_start: str, time_end: str) -> None:
    """
    Download Daymet data for a specified region, variable, and time range.

    Args:
        region (str): The region for which to download the data (e.g., 'na' for North America).
        variable (str): The Daymet variable to download (e.g., 'tmax', 'tmin', 'prcp').
        year (int): The year for which to download the data.
        boundaries (dict): The latitudinal and longitudinal boundaries of the area.
        time_start (str): The start time for the data in yyyy-mm-dd format.
        time_end (str): The end time for the data in yyyy-mm-dd format.
    """
    url = f"https://thredds.daac.ornl.gov/thredds/ncss/grid/ornldaac/2129/daymet_v4_daily_{region}_{variable}_{year}.nc"
    params = {
        "var": "lat",
        "var": "lon",
        "var": variable,
        "north": boundaries["max_lat"],
        "south": boundaries["min_lat"],
        "east": boundaries["max_lon"],
        "west": boundaries["min_lon"],
        "disableProjSubset": "on",
        "horizStride": 1,
        "time_start": f"{time_start}T00:00:00Z",
        "time_end": f"{time_end}T00:00:00Z",
        "timeStride": 1,
        "accept": "netcdf"
    }

    response = requests.get(url, params=params)
    if response.status_code == 200:
        filename = f"daymet_{variable}_{year}.nc"
        with open(filename, 'wb') as file:
            file.write(response.content)
        print(f"Data downloaded successfully: {filename}")
    else:
        print(f"Failed to download data: {response.status_code}")

# Define the boundaries for Gambia
GAMBIA_BOUNDARIES = {
    "min_lat": 13.052113101601,
    "max_lat": 13.837920583180,
    "min_lon": -17.06499856438,
    "max_lon": -13.77921401885
}

In [3]:
# Example usage for the year 2015, for the 'tmax' variable
download_daymet_data(
    region="na",
    variable="tmax",
    year=2015,
    boundaries=GAMBIA_BOUNDARIES,
    time_start="2015-01-01",
    time_end="2015-12-31"
)


Data downloaded successfully: daymet_tmax_2015.nc


In [8]:
import netCDF4 as nc

def inspect_netCDF_file(file_path: str):
    """
    Print the variables and dimensions in a netCDF file.

    Args:
        file_path (str): Path to the netCDF file.
    """
    with nc.Dataset(file_path, mode='r') as dataset:
        print("Variables:")
        for var in dataset.variables:
            print(var, dataset.variables[var].shape)

        print("\nDimensions:")
        for dim in dataset.dimensions:
            print(dim, len(dataset.dimensions[dim]))

# Inspeccionar el archivo netCDF
file_path = 'daymet_tmax_2015.nc'
inspect_netCDF_file(file_path=file_path)


Variables:
tmax (364, 376, 1)
time (364,)
y (376,)
x (1,)
lambert_conformal_conic ()

Dimensions:
time 364
y 376
x 1


In [10]:
import netCDF4 as nc
import pandas as pd
import numpy as np

def convert_netCDF_to_dataframe(file_path: str, variable: str) -> pd.DataFrame:
    """
    Convert a netCDF file to a pandas DataFrame.

    Args:
        file_path (str): Path to the netCDF file.
        variable (str): The variable name in the netCDF file (e.g., 'tmax').

    Returns:
        pd.DataFrame: A DataFrame containing the data from the netCDF file.
    """
    with nc.Dataset(file_path) as dataset:
        # Obtener las variables
        times = dataset.variables['time'][:]
        y = dataset.variables['y'][:]  # Suponiendo que 'y' representa latitud
        x = dataset.variables['x'][:]  # Suponiendo que 'x' representa longitud
        data = dataset.variables[variable][:]

        # Preparar los datos para el DataFrame
        time_grid, y_grid, x_grid = np.meshgrid(times, y, x, indexing='ij')
        df = pd.DataFrame({
            'time': time_grid.flatten(),
            'latitude': y_grid.flatten(),
            'longitude': x_grid.flatten(),
            variable: data.flatten()
        })

    return df

# Usar la función para convertir un archivo netCDF a DataFrame
file_path = 'daymet_tmax_2015.nc'
df = convert_netCDF_to_dataframe(file_path=file_path, variable='tmax')

# Ahora df es un DataFrame que puedes usar para machine learning


In [18]:
df

,time,latitude,longitude,tmax
0,23741.5,1575.0,3252.75,NaN
1,23741.5,1574.0,3252.75,NaN
2,23741.5,1573.0,3252.75,NaN
3,23741.5,1572.0,3252.75,NaN
4,23741.5,1571.0,3252.75,NaN
...,...,...,...,...
136859,24104.5,1204.0,3252.75,NaN
136860,24104.5,1203.0,3252.75,NaN
136861,24104.5,1202.0,3252.75,NaN
136862,24104.5,1201.0,3252.75,NaN


In [19]:
def unique_values_percentage_with_nan(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Calculate the unique values and their percentage of occurrence in a DataFrame column,
    including NaN values.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.
        column (str): The column name for which to calculate unique values and percentages.

    Returns:
        pd.DataFrame: A DataFrame with unique values, NaN counts, and their percentage of occurrence.
    """
    # Calcula el conteo de valores únicos y NaN
    value_counts = df[column].value_counts(dropna=False)

    # Calcula el número de NaNs y su porcentaje
    nan_count = df[column].isna().sum()
    nan_percentage = (nan_count / len(df)) * 100

    # Calcula el porcentaje de cada valor único
    percentage = (value_counts / len(df)) * 100

    # Crea un DataFrame para mostrar los resultados
    results_df = pd.DataFrame({
        'Unique Values or NaN': value_counts.index,
        'Counts': value_counts.values,
        'Percentage': percentage.values
    })

    # Agrega la información de NaN al principio del DataFrame
    nan_info_df = pd.DataFrame({
        'Unique Values or NaN': ['NaN'],
        'Counts': [nan_count],
        'Percentage': [nan_percentage]
    })

    return pd.concat([nan_info_df, results_df], ignore_index=True)

# Ejemplo de uso con tu DataFrame y la columna 'tmax'
tmax_unique_values_df = unique_values_percentage_with_nan(df=df, column='tmax')
print(tmax_unique_values_df)


     Unique Values or NaN  Counts  Percentage
0                     NaN  135408   98.936170
1                     NaN  135408   98.936170
2                    4.15       6    0.004384
3                    7.48       6    0.004384
4                   15.66       6    0.004384
...                   ...     ...         ...
1023                10.76       1    0.000731
1024                10.73       1    0.000731
1025                10.31       1    0.000731
1026                10.47       1    0.000731
1027                 -3.4       1    0.000731

[1028 rows x 3 columns]
